# What Is a Quantum Group? -- Stakeholder Edition

**Phase 5c, Wave 1** | SK-EFT Hawking Project -- Stakeholder Edition

## What This Is About

We have verified the **Hopf algebra** structure of the quantum group $U_q(\mathfrak{sl}_2)$.
A Hopf algebra is a mathematical object that knows how to do three things:

1. **Combine** representations (coproduct $\Delta$) -- like adding angular momenta
2. **Forget** to a scalar (counit $\varepsilon$) -- the trivial representation
3. **Invert** representations (antipode $S$) -- like taking duals

These three operations are the minimum structure needed to build a **tensor category**
of representations -- which is the mathematical framework for gauge theories.

In [ ]:
import numpy as np
from src.core.formulas import (
    uqsl2_coproduct, uqsl2_counit, uqsl2_antipode, uqsl2_antipode_squared
)
from src.core.visualizations import COLORS

## Why Does the Hopf Structure Matter?

In ordinary quantum mechanics, combining two spin-1/2 particles gives spin-0 and spin-1.
The rule for combining is encoded in **Clebsch-Gordan coefficients**. But where does this
rule come from mathematically?

The answer is the **coproduct** $\Delta$. It tells us how each symmetry generator acts on
a pair of particles. For ordinary rotations:

$$\Delta(J) = J \otimes 1 + 1 \otimes J \quad \text{(add angular momenta)}$$

For the quantum group, the coproduct is **asymmetric**:

$$\Delta(E) = E \otimes K + 1 \otimes E$$

That extra $K$ factor means the order of particles matters -- particle 1 "knows about"
particle 2 through $K$. This asymmetry is what makes quantum groups **quantum**: they
encode non-trivial braiding statistics.

In [ ]:
# Demonstrate the asymmetry: Delta is not cocommutative
q = np.exp(1j * np.pi / 5)
K = np.diag([q, q**(-1)])
Kinv = np.diag([q**(-1), q])
E = np.array([[0, 1], [0, 0]], dtype=complex)
F = np.array([[0, 0], [1, 0]], dtype=complex)
I2 = np.eye(2, dtype=complex)

tensor = lambda a, b: np.kron(a if isinstance(a, np.ndarray) else a * I2,
                               b if isinstance(b, np.ndarray) else b * I2)

delta_E = uqsl2_coproduct('E', E, F, K, Kinv, tensor)
# The "swapped" coproduct: tau o Delta
delta_E_swap = tensor(I2, E) @ tensor(K, I2) + tensor(E, I2)

print('Is the coproduct symmetric (cocommutative)?')
print(f'  ||Delta(E) - tau(Delta(E))|| = {np.linalg.norm(delta_E - delta_E_swap):.4f}')
print(f'  Answer: {"Yes (classical)" if np.allclose(delta_E, delta_E_swap) else "No -- this is a QUANTUM group"}')
print(f'\nThis asymmetry is why particle exchange creates non-trivial braiding.')

## The Three Parts of a Hopf Algebra

Think of a Hopf algebra as having three "muscles":

| Muscle | Symbol | What It Does | Analogy |
|--------|--------|-------------|--------|
| Coproduct | $\Delta$ | Combines two systems | Adding angular momenta |
| Counit | $\varepsilon$ | Projects to trivial | "Forgetting" the symmetry |
| Antipode | $S$ | Creates the dual | Taking the complex conjugate |

These must satisfy compatibility conditions (axioms). The key one:
applying $S$ to one half of the coproduct and multiplying gives the counit.
Physically: tensoring a representation with its dual and projecting gives the trivial.

In [ ]:
# Show all three structures working together
print('The three Hopf structures on U_q(sl_2) generators:')
print('=' * 55)

for gen in ['E', 'F', 'K', 'Kinv']:
    eps = uqsl2_counit(gen)
    s = uqsl2_antipode(gen, E, F, K, Kinv)
    norm_s = np.linalg.norm(s)
    print(f'  {gen:>4}:  eps = {eps},  ||S|| = {norm_s:.4f}')

print(f'\nCounit: E,F map to 0 (they change quantum numbers).')
print(f'         K,K^-1 map to 1 (they just label states).')

## The Squared Antipode: Why $S^2 \neq \mathrm{id}$

For ordinary groups, applying the "inverse" operation twice gives you back the original:
$S^2 = \mathrm{id}$. But for quantum groups:

$$S^2(x) = K x K^{-1}$$

Applying $S$ twice does not return to the original -- it **conjugates** by $K$.
This is a direct measure of how far $U_q(\mathfrak{sl}_2)$ is from being classical.

- $S^2(E) = q^2 E$ (phase shift by $q^2$)
- $S^2(F) = q^{-2} F$ (phase shift by $q^{-2}$)
- $S^2(K) = K$ (Cartan element is fixed)

At $q = 1$: $S^2 = \mathrm{id}$ (classical). At $q \neq 1$: the phase shifts are
non-trivial -- this is the mathematical origin of **anyonic statistics**.

In [ ]:
# Show how S^2 eigenvalues depend on q
print('S^2 eigenvalues as a function of q:')
print(f'{"q":>20}  {"S^2(E)/E":>14}  {"S^2(F)/F":>14}  {"S^2(K)/K":>14}')
print('-' * 68)

for label, qval in [('1 (classical)', 1.0), ('sqrt(2)', np.sqrt(2)),
                      ('exp(i*pi/6)', np.exp(1j*np.pi/6)),
                      ('exp(i*pi/3)', np.exp(1j*np.pi/3)),
                      ('i', 1j)]:
    q2 = complex(qval)**2
    qm2 = complex(qval)**(-2)
    fmt = lambda v: f'{v.real:+.3f}{v.imag:+.3f}i' if abs(v.imag) > 1e-10 else f'{v.real:.4f}'
    print(f'{label:>20}  {fmt(q2):>14}  {fmt(qm2):>14}  {"1.0000":>14}')

print(f'\nAt q=1, all eigenvalues are 1 (S^2 = id, classical).')
print(f'At q != 1, E and F pick up phases -- this encodes braiding.')

## The Chain: From Lattice to Gauge Theory

The Hopf algebra structure is the critical link in our chain:

1. **Onsager algebra** (Phase 5b) -- integrable lattice models
2. **q-deformation** (Phase 5b) -- introduces quantum parameter $q$
3. **Hopf algebra** (this notebook) -- enables tensor products of representations
4. **Fusion categories** (next) -- at roots of unity, reps become finite and topological
5. **Gauge emergence** (Phase 5b) -- Drinfeld center gives gauge theory

Without the Hopf structure (step 3), we cannot form tensor products, and without
tensor products, there are no fusion categories, and without fusion categories,
gauge forces cannot emerge. The coproduct $\Delta$ is the engine that drives
the entire construction.

**Formal verification:** All of this is proved in Lean 4, building on Mathlib's
`FreeAlgebra` and `RingQuot` infrastructure. Zero axioms -- every structure is constructed,
not assumed.